# Step 03. 사전 정의 샘플 기반 경로 계획 (Pre-defined Sample-based Planner)

> **참고**: PythonRobotics (https://github.com/AtsushiSakai/PythonRobotics), original author Atsushi Sakai (@Atsushi_twi) / License MIT 코드 참고
>
> **교육 주관**: Hanyang University, Automotive Intelligence Laboratory (AILAB)

미리 정해진 패턴(predefined pattern)으로 여러 경로 후보를 만들고, 그중 비용이 가장 낮은 안전한 후보 하나를 고르는 경로 계획입니다. 이 노트북에서는 종방향 속도 프로파일을 하나로 고정하고, 횡방향으로 여러 목표 위치를 샘플링해 후보를 만든 뒤, 비용과 충돌 검사를 거쳐 최적 경로를 선택합니다.

## 학습 목표

1. Frenet frame: 경로를 기준으로 한 $(s, d)$ 좌표 이해
2. 종방향 $s(t)$ 프로파일 고정 (등속 CV)
3. 횡방향 목표 offset $d_i$ 를 샘플링해 5차 다항식으로 후보 경로 생성
4. Frenet $\rightarrow$ Cartesian 변환으로 후보를 도로 위에 표현 (utils 변환기 사용)
5. 비용 함수와 충돌 검사로 후보를 평가하고 최적 경로 선택

## 0. Setup & 시나리오

기준 경로(reference path)는 waypoint 를 cubic spline 으로 보간해 만듭니다. 도로 위에 장애물 몇 개를 두고, 차량의 초기 상태(시작 위치와 횡방향 offset)와 샘플링과 비용 파라미터를 한곳에 모읍니다.

In [ ]:
%matplotlib inline
import sys, os
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('../utils/02_trajectory_planning'))

import numpy as np
import matplotlib.pyplot as plt

from step03_predefined_sample_based_planner import (
    ReferencePath, FrenetPathToCartesian, ShowRoad,
    PlotFrenetFrameConcept, PlotLongitudinalProfile,
    PlotLateralCandidatesFrenet, PlotCandidatesCartesian,
    PlotCandidatesCost, PlotValidity, PlotBest,
)

In [ ]:
# ----- 기준 경로 (waypoint -> cubic spline) -----
WP_X = [0.0, 20.0, 40.0, 60.0, 80.0]
WP_Y = [0.0,  3.0,  0.0, -3.0,  0.0]
ref = ReferencePath(WP_X, WP_Y, ds=0.1)

# ----- 장애물 (x, y) 와 안전 반경 -----
OBSTACLES = [[22.0, 1.5], [40.0, 1.0], [58.0, -2.5]]
ROBOT_RADIUS = 1.5

# ----- 횡방향 샘플링과 s 격자 -----
MAX_OFFSET  = 6.0   # 좌우 최대 offset [m]
OFFSET_STEP = 1.0   # offset 샘플 간격 [m]
DS          = 0.5   # 경로 길이 s 격자 간격 [m] (이산화)

# ----- 차량 초기 상태 -----
S_START   = 0.0         # 시작 위치의 경로 거리 s (arc length, 시작점이면 0)
CUR_D     = 3.0         # 현재 횡방향 위치 d (기준 경로 왼쪽 3m 에서 주행 중)
CUR_D_D   = 0.0         # 현재 횡방향 기울기 d'
CUR_D_DD  = 0.0         # 현재 횡방향 곡률 d''
V         = 30.0 / 3.6  # 종방향 등속 [m/s]
T_HORIZON = 6.0         # 계획 구간 시간 [s]

# ----- 비용 가중치 -----
K_J = 0.1              # 횡방향 jerk 비용
K_D = 1.0              # 목표 offset 비용

print(f'path length = {ref.length:.1f} m,  horizon s in [{S_START:.0f}, {S_START + V * T_HORIZON:.0f}] m')

In [ ]:
# 시작 위치(기준점에서 횡방향으로 CUR_D 만큼 떨어진 곳)
start_x = ref.x[0] - np.sin(ref.yaw[0]) * CUR_D
start_y = ref.y[0] + np.cos(ref.yaw[0]) * CUR_D

ShowRoad(ref, OBSTACLES, ROBOT_RADIUS, start=(start_x, start_y))

## 1. Frenet frame: 경로를 기준으로 한 $(s, d)$ 좌표

도로처럼 구조화된 환경에서는 $x, y$ 대신 **기준 경로(reference path)** 를 따라 위치를 보는 것이 편합니다. 이 실습의 기준 경로는 도로 중앙선(강의 자료의 road centerline)이지만, 전역 계획이 만든 경로처럼 다른 선도 기준이 될 수 있습니다. 경로 위의 한 점을 두 값으로 나타냅니다.

- $s$ : 경로를 따라 잰 거리(arc length), 즉 "얼마나 멀리 갔는가"
- $d$ : 경로에서 수직으로 떨어진 횡방향 거리, 즉 "기준 경로에서 얼마나 옆에 있는가" (왼쪽 +)

이렇게 보면 "차선을 따라가며 옆으로 얼마나 비킬까"를 그대로 $d$ 로 표현할 수 있어, 횡방향 회피 경로를 만들기 쉬워집니다.

아래 그림은 **같은 한 점**을 (a) 굽은 Cartesian 경로와 (b) 곧게 편 Frenet $(s, d)$ 평면에서 비교합니다. 경로를 곧게 펴면 $(s, d)$ 가 그대로 가로와 세로 좌표가 되어, 횡방향 회피를 직관적으로 다룰 수 있습니다.

In [ ]:
PlotFrenetFrameConcept(ref, s_query=15.0, d_query=3.0)

## 2. 종방향 $s(t)$ 프로파일 고정

이번 실습에서는 "얼마나 빨리 나아갈지"를 정하는 종방향 프로파일 $s(t)$ 를 **하나로 고정**합니다. 가장 단순한 등속(CV) 프로파일을 씁니다.

$$ s(t) = s_{\text{start}} + v\,t $$

(가속 프로파일(CA)도 같은 틀로 확장할 수 있지만, 여기서는 CV 하나만 사용합니다.)

종방향을 고정했으므로 후보는 **횡방향**에서만 갈라집니다. 계획 구간의 끝 경로 거리는 $s_{\text{end}} = s_{\text{start}} + v\,T$ 이고, 이 구간을 $\Delta s$ 간격의 점으로 **이산화**합니다. (섹션 3 에서 목표 offset 을 고르는 "샘플링"과는 다릅니다. 이것은 곡선을 계산하고 그리기 위한 격자일 뿐이며, 모든 후보가 같은 s 격자를 공유합니다.)

> 일반적인 Frenet planner 는 $s(t)$ 와 $d(s)$ 를 모두 샘플링할 수 있지만, 여기서는 핵심인 횡방향 후보 생성에 집중하기 위해 종방향을 고정합니다.

In [ ]:
s_end = S_START + V * T_HORIZON
s_arr = np.arange(S_START, s_end + DS/2, DS)   # 끝점 s_end 까지 포함 (마지막 점이 목표 offset 에 정확히 안착)
print(f's_start = {S_START:.1f} m,  s_end = {s_end:.1f} m,  s 격자 점 수 = {len(s_arr)}')

PlotLongitudinalProfile(S_START, V, T_HORIZON)

## 3. 횡방향 후보 생성: 5차 다항식

고정된 종방향에 대해, 여러 **목표 lateral offset $d_i$** 를 샘플링합니다. 각 $d_i$ 마다 현재 횡방향 상태에서 그 offset 까지 **부드럽게** 이어지는 곡선 $d(u)$ 를 하나 만듭니다. ($u = s - s_{\text{start}}$, 구간 길이 $T = s_{\text{end}} - s_{\text{start}}$)

> 주의: 여기서 $T$ 는 시간이 아니라 **거리 [m]** 입니다. 곡선이 $s$ 의 함수 $d(s)$ 라서 구간 길이도 거리로 잽니다: $T = v \cdot T_{HORIZON} = 8.33 \times 6 = 50$ m (코드의 `T_HORIZON` 6 s 와 다른 값)

### 후보 곡선의 조건

부드러움을 위해 양 끝점에서 위치, 기울기, 곡률($d, d', d''$)을 지정합니다. 조건이 6개이므로 5차 다항식을 씁니다.

$$ d(u) = a_0 + a_1 u + a_2 u^2 + a_3 u^3 + a_4 u^4 + a_5 u^5 $$

| 조건 | 시작점 $u=0$ | 끝점 $u=T$ |
|---|---|---|
| 위치 $d$ | $d(0) = d_{\text{cur}}$ | $d(T) = d_i$ |
| 1차 미분 $d'$ | $d'(0) = d_{\text{cur}}'$ | $d'(T) = 0$ |
| 2차 미분 $d''$ | $d''(0) = d_{\text{cur}}''$ | $d''(T) = 0$ |

- **시작점 조건 ($u=0$): 차량의 현재 상태, 주어지는 값입니다.** 새 경로는 차가 지금 있는 위치($d_{\text{cur}}$), 지금의 진행 방향(기울기 $d_{\text{cur}}'$), 지금의 곡률($d_{\text{cur}}''$: 기준 경로에 대해 휘는 정도, 현재 조향과 관련)에서 시작해야 현재 운동에 매끄럽게 이어집니다. 하나라도 어긋나면 위치가 점프하거나, 방향이 꺾이거나, 조향을 순간적으로 돌려야 합니다. step01 에서 배운 접점의 C0/C1/C2 연속과 같은 개념입니다.
- **끝점 조건 ($u=T$): 우리가 고르는 값입니다.** 끝 위치는 목표 offset $d_i$ 로 둡니다. $d'(T) = 0$ 이면 목표선과 평행하게 도착하고, $d''(T) = 0$ 이면 도착 순간 기준 경로와 같은 모양으로 휘며 달리고 있어서 그대로 목표선(기준 경로에서 $d_i$ 만큼 떨어진 선)을 따라갈 수 있습니다. (기준 경로가 직선이면 이때 조향이 중립)

### 계수 구하기

$d(u)$ 를 미분하면 (거듭제곱 법칙 $\frac{d}{du}u^k = k\,u^{k-1}$):

$$
\begin{aligned}
d'(u)  &= a_1 + 2 a_2 u + 3 a_3 u^2 + 4 a_4 u^3 + 5 a_5 u^4 \\
d''(u) &= 2 a_2 + 6 a_3 u + 12 a_4 u^2 + 20 a_5 u^3
\end{aligned}
$$

**시작점 $u=0$** 을 넣으면 $u$ 가 붙은 항이 모두 사라져 계수가 바로 나옵니다.

$$ a_0 = d_{\text{cur}}, \qquad a_1 = d_{\text{cur}}', \qquad a_2 = d_{\text{cur}}'' / 2 $$

**끝점 $u=T$** 에서는 세 식에 $T$ 를 대입하고 끝조건 $(d_i, 0, 0)$ 을 겁니다. 이미 아는 $a_0, a_1, a_2$ 항을 우변으로 넘기면, 미지수 $a_3, a_4, a_5$ 에 대한 **3x3 연립방정식**이 됩니다. 각 식에서 미지수에 붙는 **계수가 그대로 행렬 $A$** 입니다.

$$
\underbrace{\begin{bmatrix} T^3 & T^4 & T^5 \\ 3T^2 & 4T^3 & 5T^4 \\ 6T & 12T^2 & 20T^3 \end{bmatrix}}_{A}
\begin{bmatrix} a_3 \\ a_4 \\ a_5 \end{bmatrix}
=
\begin{bmatrix} d_i - (a_0 + a_1 T + a_2 T^2) \\ 0 - (a_1 + 2 a_2 T) \\ 0 - 2 a_2 \end{bmatrix}
$$

- **각 행** = 끝점 조건 하나: 1행 위치 $d(T)$, 2행 기울기 $d'(T)$, 3행 곡률 $d''(T)$.
- **각 열** = 미지수 하나 ($a_3, a_4, a_5$); 한 칸은 "그 식에서 그 미지수에 붙는 계수". 예) 1행은 $d(T)$ 의 $a_3 u^3, a_4 u^4, a_5 u^5$ 에 $u=T$ 를 넣은 $T^3, T^4, T^5$.
- **세로로 내려갈수록 한 번씩 미분**: $a_3$ 열 $T^3 \to 3T^2 \to 6T$, $a_4$ 열 $T^4 \to 4T^3 \to 12T^2$, $a_5$ 열 $T^5 \to 5T^4 \to 20T^3$ (2행은 1행을, 3행은 2행을 미분).

### 실습 (TODO 1~3)

아래 코드에서 세 가지를 직접 채웁니다: **TODO 1 목표 offset 샘플링**, **TODO 2 계수 행렬 $A$**, **TODO 3 끝점 경계조건**.

In [ ]:
##### TODO 1 #####
# target_offsets 만들기: 샘플링할 목표 offset d_i 배열
#   - 범위: -MAX_OFFSET ~ +MAX_OFFSET (양 끝 포함)
#   - 간격: OFFSET_STEP
# 사용함수: np.arange(start, stop, step) / 기능: start 이상 stop "미만"을 step 간격으로
#################

target_offsets = ...

#################

print(f'목표 offset d_i = {[round(d, 1) for d in target_offsets]}')

# 확인: -MAX_OFFSET ~ +MAX_OFFSET (양 끝 포함), OFFSET_STEP 간격
assert np.isclose(target_offsets[0], -MAX_OFFSET) and np.isclose(target_offsets[-1], MAX_OFFSET) \
    and np.allclose(np.diff(target_offsets), OFFSET_STEP), \
    'TODO 1 확인 - 양 끝 포함 필요: np.arange 의 stop 은 "미만" -> stop = MAX_OFFSET + OFFSET_STEP'

In [ ]:
class Candidate:
    """하나의 횡방향 후보 경로 (Frenet 곡선 + 이후 채울 Cartesian/비용/유효성)."""
    def __init__(self, di, s, d, d_d, d_dd, d_ddd):
        self.di = di                                                # 목표 lateral offset
        self.s, self.d = s, d                                       # Frenet: 경로 거리 s, 횡 offset d(s)
        self.d_d, self.d_dd, self.d_ddd = d_d, d_dd, d_ddd          # d 의 s 에 대한 1/2/3차 미분
        self.x = self.y = self.yaw = self.kappa = self.v = None     # 도로 위 Cartesian 궤적 (위치 / 방향 / 곡률 / 속도)
        self.cost = None                                            # 경로 비용 (작을수록 좋은 후보)
        self.valid = True                                           # 충돌 검사 통과 여부


def QuinticCoeffs(start_d, start_d_d, start_d_dd, end_d, end_d_d, end_d_dd, T):
    """양 끝 경계조건 6개로 5차 다항식 d(u)=a0+...+a5 u^5 의 계수를 구한다.

    시작점 u=0 의 세 조건은 a0, a1, a2 를 바로 준다.
    끝점 u=T 의 세 조건은 a3, a4, a5 에 대한 3x3 연립방정식 A x = b 로 푼다.

    인자:
      start_d, start_d_d, start_d_dd : 시작점(u=0)의 횡위치 d, 기울기 d'(= dd/ds), 곡률 d''
      end_d, end_d_d, end_d_dd       : 끝점(u=T)의 횡위치 d, 기울기 d', 곡률 d''
      T                              : 구간 길이 (= s_end - s_start)
    """
    a0 = start_d           # d(0)   = a0
    a1 = start_d_d         # d'(0)  = a1
    a2 = start_d_dd / 2.0  # d''(0) = 2 a2

    ##### TODO 2 #####
    # 계수 행렬 A (3x3) 채우기: markdown 의 연립방정식에 있는 A 그대로
    #   - 각 행 = 끝점 조건 하나  (1행 d(T) / 2행 d'(T) / 3행 d''(T))
    #   - 각 열 = 미지수 하나    (a3 / a4 / a5)
    # 사용함수: np.array(list) / 기능: 중첩 리스트 -> 행렬
    #################

    A = ...

    #################

    # 우변 b = 끝 조건 - (이미 정해진 a0,a1,a2 가 만드는 몫)
    b = np.array([end_d    - (a0 + a1*T + a2*T**2),
                  end_d_d  - (a1 + 2*a2*T),
                  end_d_dd - 2*a2])
    a3, a4, a5 = np.linalg.solve(A, b)
    return np.array([a0, a1, a2, a3, a4, a5])


def EvalPoly(coeffs, u):
    """다항식 값과 1, 2, 3차 미분을 u(= s - s_start, 구간 내 위치 배열)에서 평가."""
    a0, a1, a2, a3, a4, a5 = coeffs
    d     = a0 + a1*u + a2*u**2 + a3*u**3 + a4*u**4 + a5*u**5
    d_d   = a1 + 2*a2*u + 3*a3*u**2 + 4*a4*u**3 + 5*a5*u**4
    d_dd  = 2*a2 + 6*a3*u + 12*a4*u**2 + 20*a5*u**3
    d_ddd = 6*a3 + 24*a4*u + 60*a5*u**2
    return d, d_d, d_dd, d_ddd

In [ ]:
def GenerateLateralCandidates(s_arr, s_start, s_end, cur_d, cur_d_d, cur_d_dd):
    """target_offsets 의 목표 offset d_i 마다 5차 다항식 후보 하나씩 생성한다.

    인자:
      s_arr                    : 경로 거리 s 격자 배열
      s_start, s_end           : 구간 시작/끝 s  (u=0 이 s_start, u=T 가 s_end)
      cur_d, cur_d_d, cur_d_dd : 차량의 현재 횡방향 상태 (위치 d, 기울기 d', 곡률 d'')
    """
    u = s_arr - s_start  # 구간 내 위치 (0 <= u <= T)
    T = s_end - s_start  # 구간 길이
    candidates = []
    for di in target_offsets:
        # 시작점 u=0 : 차량의 현재 횡방향 상태 그대로
        start_d, start_d_d, start_d_dd = cur_d, cur_d_d, cur_d_dd

        ##### TODO 3 #####
        # 끝점 u=T 의 경계조건 3개 정하기 (경계조건 표 참고)
        #   - end_d    (끝 위치)   : 목표 offset 으로
        #   - end_d_d  (끝 기울기) : 목표선에 평행하게 도착하려면?
        #   - end_d_dd (끝 곡률)   : 도착 후 목표선을 그대로 따라가려면?
        #################

        end_d, end_d_d, end_d_dd = ...

        #################

        coeffs = QuinticCoeffs(start_d, start_d_d, start_d_dd, end_d, end_d_d, end_d_dd, T)
        d, d_d, d_dd, d_ddd = EvalPoly(coeffs, u)
        candidates.append(Candidate(di, s_arr, d, d_d, d_dd, d_ddd))
    return candidates


candidates = GenerateLateralCandidates(s_arr, S_START, s_end, CUR_D, CUR_D_D, CUR_D_DD)
print(f'생성된 후보 수 = {len(candidates)}')

# 확인: 모든 후보의 끝점 (d, d', d'') = (d_i, 0, 0)  (행렬 A 나 끝점 조건이 틀리면 어긋남)
for c in candidates:
    assert np.allclose([c.d[-1], c.d_d[-1], c.d_dd[-1]], [c.di, 0.0, 0.0], atol=1e-6), \
        (f"TODO 2·3 확인 - d_i = {c.di:+.0f} 후보의 끝점 (d, d', d'') = ({c.d[-1]:.2f}, {c.d_d[-1]:.3f}, {c.d_dd[-1]:.4f}), "
         f"기대값 ({c.di:.0f}, 0, 0) -> 행렬 A 의 각 행, 끝점 조건 확인")
print("✅ 경계조건 확인 통과: 모든 후보의 끝점 (d, d', d'') = (d_i, 0, 0)")

In [ ]:
PlotLateralCandidatesFrenet(candidates)

## 4. 후보를 도로 위로: Frenet $\rightarrow$ Cartesian

Frenet 좌표 $(s, d)$ 로 만든 후보를 실제 도로 위 $(x, y)$ 로 옮깁니다. 변환에는 `FrenetConverter` 를 사용합니다 (`FrenetPathToCartesian` 이 내부에서 호출). 변환하면 위치뿐 아니라 방향, 곡률, 속도도 함께 얻어 다음 단계의 제약 검사에 쓸 수 있습니다.

In [ ]:
for cand in candidates:
    cartesian = FrenetPathToCartesian(ref, cand.s, cand.d, cand.d_d, cand.d_dd,
                                      s_d=V, s_dd=0.0)
    cand.x, cand.y = cartesian['x'], cartesian['y']
    cand.yaw, cand.kappa, cand.v = cartesian['yaw'], cartesian['kappa'], cartesian['v']

PlotCandidatesCartesian(ref, OBSTACLES, ROBOT_RADIUS, candidates,
                        start=(start_x, start_y))

## 5. 비용 함수: 어떤 후보가 좋은가

후보마다 비용을 매겨 "좋음"을 수치화합니다. 여기서는 두 가지를 봅니다.

- **부드러움**: 횡방향 jerk(가속도가 바뀌는 빠르기 = 시간 기준 3차 미분 $d'''(t)$) 제곱합이 작을수록 좋음 (급격한 조향 변화를 피함)
- **기준 경로 유지 (offset 비용)**: 최종 offset $d_i$ 가 0(기준 경로)에 가까울수록 좋음 (불필요하게 멀리 비키지 않음)

후보 곡선은 $s$ 의 함수 $d(s)$ 이지만, 승차감은 **시간** 기준 jerk 로 느낍니다. 종방향이 등속이라 $s = v\,t$ 이므로 $d'''(t) = v^3\, d'''(s)$ 로 바꿔 씁니다.

$$ \text{cost} = K_J \sum \big(v^3\, d'''(s)\big)^2 + K_D \, d_i^2 $$

지금 차는 기준 경로 **왼쪽 3m** 에서 달리고 있습니다. 기준 경로 쪽으로 돌아가면 offset 비용은 줄지만 조향(jerk)이 필요하고, 그대로 가면 jerk 는 없지만 offset 비용을 냅니다. 두 항이 서로 반대 방향으로 당기는 **트레이드오프**이고, 그 균형을 $K_J, K_D$ 가 정합니다.

In [ ]:
def ComputeCost(candidates):
    for cand in candidates:
        jerk = cand.d_ddd * V ** 3   # d'''(s) -> 시간 기준 jerk d'''(t)  (s = V·t)

        ##### TODO 4 #####
        # 후보의 비용 계산하기
        #   비용 = K_J * (jerk 제곱합) + K_D * (목표 offset 제곱)
        # 사용함수: np.sum(np.ndarray) / 기능: 배열 원소의 총합
        #################

        cand.cost = ...

        #################


ComputeCost(candidates)

# 비용을 두 항으로 분해해 확인 (jerk 항 = cost 에서 offset 항을 뺀 나머지)
print(f'{"d_i":>5}   {"jerk 항":>9}   {"offset 항":>9}   {"cost":>7}')
for cand in candidates:
    offset_term = K_D * cand.di ** 2
    jerk_term = cand.cost - offset_term
    print(f'{cand.di:+5.1f}   {jerk_term:9.2f}   {offset_term:9.2f}   {cand.cost:7.2f}')

# 확인: K_J = 0.1, K_D = 1.0 이면 d_i = 0 -> 1.46, d_i = +3 (jerk 0, 지금 offset 유지) -> 9.00
_cost = {round(c.di): c.cost for c in candidates}
print('✅ cost 확인 통과' if abs(_cost.get(0, np.nan) - 1.46) < 0.01 and abs(_cost.get(3, np.nan) - 9.00) < 0.01
      else '❌ 기대값 d_i=0 -> 1.46, d_i=+3 -> 9.00 -> TODO 4 확인 (K_J·K_D 를 바꿨다면 무시)')

PlotCandidatesCost(ref, OBSTACLES, ROBOT_RADIUS, candidates,
                   start=(start_x, start_y))

## 6. 제약과 충돌 검사: 갈 수 있는 후보만 남기기

비용이 낮아도 장애물과 부딪히면 쓸 수 없습니다. 각 후보의 경로 점이 어떤 장애물과도 안전 반경 안으로 들어오지 않는지 검사해, 통과한 후보만 유효(valid)로 표시합니다. (속도와 곡률 한계 같은 다른 제약도 같은 자리에서 함께 검사할 수 있습니다.)

In [ ]:
def CheckCollision(cand, obstacles, robot_radius):
    for ox, oy in obstacles:
        dist2 = (cand.x - ox) ** 2 + (cand.y - oy) ** 2
        if np.any(dist2 <= robot_radius ** 2):
            return False        # 한 점이라도 반경 안 -> 충돌
    return True


for cand in candidates:
    cand.valid = CheckCollision(cand, OBSTACLES, ROBOT_RADIUS)

n_valid = sum(c.valid for c in candidates)
print(f'유효 후보 = {n_valid} / {len(candidates)}')
PlotValidity(ref, OBSTACLES, ROBOT_RADIUS, candidates, start=(start_x, start_y))

## 7. Best candidate 선택

유효한 후보 중에서 비용이 가장 낮은 하나를 최종 경로로 고릅니다. 이것이 sample-based planner 의 결론입니다: 미리 만든 후보들 중 "안전하면서 가장 비용이 낮은" 것.

In [ ]:
def SelectBest(candidates):
    best, best_cost = None, float('inf')
    for cand in candidates:
        ##### TODO 5 #####
        # best 갱신하기: 유효(valid)하면서 cost 가 지금까지의 최솟값보다 작을 때
        #   - 갱신 대상: best 와 best_cost 둘 다
        #################

        ...

        #################
    return best


best = SelectBest(candidates)
if best is None:
    print('best 없음: 모든 후보가 충돌했거나, TODO 5 에서 best 를 갱신하지 않음')
else:
    print(f'best: 목표 offset d_i = {best.di:+.1f} m,  cost = {best.cost:.2f}')
    blocked = [f'{c.di:+.0f}' for c in candidates if c.cost < best.cost and not c.valid]
    if blocked:
        print(f'  (cost 가 더 낮은 d_i = {", ".join(blocked)} m 는 충돌 후보라 제외)')
    # 확인: 기본 설정이면 best d_i = -1.0 m
    print('✅ best 확인 통과' if np.isclose(best.di, -1.0)
          else '❌ 기대값 d_i = -1.0 m -> TODO 5 조건 (valid 이고 cost < best_cost) 확인 (설정을 바꿨다면 무시)')
    PlotBest(ref, OBSTACLES, ROBOT_RADIUS, candidates, best, start=(start_x, start_y))

### 해보기: 부드러움 가중치 $K_J$ 를 키우면?

비용 가중치만 바꿔도 고르는 경로가 달라집니다. $K_J$ 가 작으면 조향을 감수하고 기준 경로 쪽으로 돌아오고, 크면 조향을 아껴 지금 offset 을 그대로 유지합니다.

In [ ]:
# K_J (부드러움 가중치) 만 바꿔 best 후보가 어떻게 바뀌는지 비교
for k_j in [0.1, 1.0]:
    K_J = k_j
    ComputeCost(candidates)
    b = SelectBest(candidates)
    print(f'K_J = {k_j:4.1f}  ->  best d_i = {b.di:+.1f} m  (cost {b.cost:.2f})')

K_J = 0.1                 # 원래 값으로 복구
ComputeCost(candidates)

---

## 정리

- 종방향을 고정하고 횡방향 목표 offset $d_i$ 만 샘플링해, 각 후보를 5차 다항식 $d(s)$ 로 부드럽게 생성했습니다.
- Frenet $\rightarrow$ Cartesian 변환으로 후보를 도로 위에 옮기고, 비용과 충돌 검사로 평가해 최적 경로를 선택했습니다.
- 이 흐름이 "predefined pattern 으로 후보 생성 $\rightarrow$ 최소 비용 후보 선택" 이라는 sample-based planner 의 핵심입니다.